In [ ]:
!pip install -U transformers

In [ ]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
senquences = ["I've been waitting for a Hugging Face Course whole my life.",
                 "This course is amazing!"]

batch = tokenizer(senquences, padding = True, truncation = True, return_tensors = "pt")
batch["labels"] = torch.tensor([1,1])
optimizer = AdamW(model.parameters())

loss = model(**batch).loss
loss.backward()
optimizer.step()

# load Dataset

In [ ]:
from datasets import load_dataset

In [ ]:
raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

In [ ]:
raw_datatrain = raw_datasets["train"]
sentences  = raw_datatrain[15]
sentences

In [ ]:
token_sentence = tokenizer(sentences["sentence1"], sentences["sentence2"])
print(token_sentence)

In [ ]:
tokenizer.convert_ids_to_tokens(token_sentence ["input_ids"])

In [ ]:
raw_datatrain.features

Processing

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)


In [ ]:
# idea in token to ids
# 1
# tokenizer_sentence1 = tokenizer(raw_datasets["train"]["sentence1"])
# tokenizer_sentence2 = tokenizer(raw_datasets["train"]["sentence2"])
# 2
# tokennize_dataset = tokenizer(raw_datasets["train"]["sentence1"], raw_datasets["train"]["sentence2"],
#                                           truncation = True, padding = True)

# good option
def tokenizer_function(sample):
  return tokenizer(sample["sentence1"], sample["sentence2"], truncation = True )

tokenizer_dataset = raw_datasets.map(tokenizer_function)
tokenizer_dataset

# Dynamic padding

In [ ]:
from transformers import DataCollatorWithPadding

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

text example:

In [ ]:
samples = tokenizer_dataset["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

In [ ]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

In [ ]:
from torch.utils.data import DataLoader

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)
for batch in dataloader:
    print({k: v.shape for k, v in batch.items()})